# Projeto IA — HAM10000: Visualizações

Material visual para o relatório:

1. **t-SNE** das features do DINOv2 — como as 7 classes se separam (ou se sobrepõem) no espaço de representação.
2. **Grad-CAM** — mapas de calor de *onde* a CNN olhou, para **ConvNeXt V2** e **EfficientNetV2-S**.
3. **Galeria de erros** — imagens classificadas erradas (real × predito).
4. **Checkpoints** — treina-ou-carrega: salva os pesos para não retreinar a cada sessão (ajuda com o limite de GPU).

> Rodar no **Colab com GPU T4**. Dica: aponte `CKPT_DIR` para o Google Drive para os checkpoints persistirem.

## 0. Setup

In [ ]:
!pip install -q timm kagglehub grad-cam

In [ ]:
import os, copy, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.manifold import TSNE
from sklearn.metrics import f1_score

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
RESULTS_DIR = 'results'; os.makedirs(RESULTS_DIR, exist_ok=True)
CKPT_DIR = '/content/checkpoints'   # p/ persistir, use algo como /content/drive/MyDrive/ham_ckpts
os.makedirs(CKPT_DIR, exist_ok=True)

## 1. Dados, split e transforms (mesmas convenções dos outros notebooks)

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
df = pd.read_csv(os.path.join(DATA_DIR, 'HAM10000_metadata.csv'))
img_paths = {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith('.jpg'):
            img_paths[os.path.splitext(f)[0]] = os.path.join(root, f)
df['path'] = df['image_id'].map(img_paths)
df = df.dropna(subset=['path']).reset_index(drop=True)
CLASSES = sorted(df['dx'].unique().tolist())
CLASS_TO_IDX = {cl: i for i, cl in enumerate(CLASSES)}
df['label'] = df['dx'].map(CLASS_TO_IDX)
MEL_IDX = CLASS_TO_IDX['mel']

def grouped_split(frame, seed=SEED):
    idx = np.arange(len(frame)); y = frame['label'].values; g = frame['lesion_id'].values
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trainval_pos, test_pos = next(sgkf.split(idx, y, g))
    tv = frame.iloc[trainval_pos].reset_index(drop=True)
    sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_pos, val_pos = next(sgkf2.split(np.arange(len(tv)), tv['label'].values, tv['lesion_id'].values))
    return (tv.iloc[tr_pos].reset_index(drop=True),
            tv.iloc[val_pos].reset_index(drop=True),
            frame.iloc[test_pos].reset_index(drop=True))

train_df, val_df, test_df = grouped_split(df)

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandAugment(num_ops=2, magnitude=7), T.ToTensor(), T.Normalize(MEAN, STD)])

class HAMDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        return self.transform(Image.open(row['path']).convert('RGB')), int(row['label'])

print(f'Treino {len(train_df)} | Val {len(val_df)} | Teste {len(test_df)}')

## 2. t-SNE das features do DINOv2

Projeta os embeddings (768-d) do conjunto de teste em 2D, coloridos por classe.
A sobreposição entre `mel` e `nv` que costuma aparecer aqui é a mesma confusão vista na matriz.

In [ ]:
dino = timm.create_model('vit_base_patch14_dinov2.lvd142m',
                         pretrained=True, num_classes=0, img_size=IMG_SIZE).eval().to(device)
for p in dino.parameters(): p.requires_grad_(False)

@torch.no_grad()
def extract_features(frame):
    loader = DataLoader(HAMDataset(frame, eval_tf), batch_size=64, shuffle=False, num_workers=2)
    feats, labels = [], []
    for x, y in loader:
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            f = dino(x.to(device))
        feats.append(f.float().cpu().numpy()); labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

print('Extraindo features do teste...')
Xte, yte = extract_features(test_df)
print('Rodando t-SNE (pode levar ~1-2 min)...')
emb = TSNE(n_components=2, init='pca', perplexity=30, random_state=SEED).fit_transform(Xte)

plt.figure(figsize=(8, 7))
for ci, cl in enumerate(CLASSES):
    m = (yte == ci)
    plt.scatter(emb[m, 0], emb[m, 1], s=7, alpha=0.6, label=cl)
plt.legend(markerscale=2, fontsize=9, loc='best')
plt.title('t-SNE das features DINOv2 (conjunto de teste)')
plt.xticks([]); plt.yticks([]); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tsne_dinov2.png'), dpi=120); plt.show()

## 3. Treina-ou-carrega os dois CNNs (com checkpoint)

Se já houver checkpoint salvo, ele só carrega. Senão, treina (mesma receita do v3: warmup → cosine, MixUp/CutMix, early stopping) e salva.

**Dica anti-limite-de-GPU:** se a sessão cair, treine **um modelo por vez** (deixe só um na lista `BACKBONES`); o outro reaproveita o checkpoint depois.

In [ ]:
BACKBONES = ['convnextv2_tiny.fcmae_ft_in22k_in1k', 'tf_efficientnetv2_s.in21k_ft_in1k']
BATCH = 32; WARMUP = 2; EPOCHS = 20; LR_HEAD = 1e-3; LR_FULL = 1e-4; PATIENCE = 4

val_loader  = DataLoader(HAMDataset(val_df, eval_tf),  batch_size=BATCH, shuffle=False, num_workers=2)
test_loader = DataLoader(HAMDataset(test_df, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)

counts = train_df['label'].value_counts().sort_index().values
inv = 1.0 / counts; sample_w = inv[train_df['label'].values]
sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double),
                                num_samples=len(train_df), replacement=True)
train_loader = DataLoader(HAMDataset(train_df, train_tf), batch_size=BATCH,
                          sampler=sampler, num_workers=2, drop_last=True)
mixup_fn = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5, switch_prob=0.5,
                 mode='batch', label_smoothing=0.05, num_classes=len(CLASSES))
criterion = SoftTargetCrossEntropy()
scaler_amp = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

def run_epoch(model, optimizer):
    model.train(); total = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device); x, y = mixup_fn(x, y)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            loss = criterion(model(x), y)
        scaler_amp.scale(loss).backward(); scaler_amp.step(optimizer); scaler_amp.update()
        total += loss.item() * x.size(0)
    return total / len(train_loader.dataset)

@torch.no_grad()
def predict_loader(model, loader, tta=False):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        x = x.to(device)
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            prob = model(x).softmax(1)
            if tta:
                prob = prob + model(torch.flip(x, dims=[3])).softmax(1)
                prob = prob + model(torch.flip(x, dims=[2])).softmax(1)
        ps.append(prob.argmax(1).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def val_macro_f1(model):
    yv, pv = predict_loader(model, val_loader, tta=False)
    return f1_score(yv, pv, average='macro')

def train_or_load(backbone):
    run_name = 'B_' + backbone.split('.')[0] + '_mixup'
    ckpt = os.path.join(CKPT_DIR, run_name + '.pt')
    model = timm.create_model(backbone, pretrained=True, num_classes=len(CLASSES)).to(device)
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        print(f'{run_name}: carregado do checkpoint.'); return model.eval(), run_name
    print(f'{run_name}: treinando...')
    best_f1, best_state, no_improve = -1.0, None, 0
    for p in model.parameters(): p.requires_grad_(False)
    for p in model.get_classifier().parameters(): p.requires_grad_(True)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD)
    for _ in range(WARMUP):
        run_epoch(model, opt); vf1 = val_macro_f1(model)
        if vf1 > best_f1: best_f1, best_state = vf1, copy.deepcopy(model.state_dict())
    for p in model.parameters(): p.requires_grad_(True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_FULL, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS - WARMUP)
    for epoch in range(WARMUP, EPOCHS):
        run_epoch(model, opt); vf1 = val_macro_f1(model); sched.step()
        print(f'  [{epoch+1}/{EPOCHS}] val macro-F1 {vf1:.3f}')
        if vf1 > best_f1: best_f1, best_state, no_improve = vf1, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE: print('  early stopping'); break
    model.load_state_dict(best_state)
    torch.save(best_state, ckpt)
    print(f'{run_name}: salvo em {ckpt} (val macro-F1 {best_f1:.3f})')
    return model.eval(), run_name

models = {}
for b in BACKBONES:
    m, name = train_or_load(b)
    models[name] = (m, b)
print('Modelos prontos:', list(models.keys()))

## 4. Grad-CAM — onde cada CNN "olhou"

Para cada modelo, mostramos uma imagem por classe: original × mapa de calor sobreposto (com a classe predita). Observe se o foco cai sobre a lesão ou sobre artefatos (pelos, régua, tinta) — possível efeito *Clever Hans*.

In [ ]:
def target_layers_for(model, backbone):
    if 'efficientnet' in backbone: return [model.conv_head]
    if 'convnext' in backbone:     return [model.stages[-1]]
    return [list(model.modules())[-1]]

def load_for_cam(path):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    rgb = np.asarray(img).astype(np.float32) / 255.0
    tens = T.Normalize(MEAN, STD)(T.ToTensor()(img)).unsqueeze(0).to(device)
    return rgb, tens

def gradcam_grid(model, backbone, run_name):
    # uma imagem (a primeira) de cada classe no teste
    rows = [test_df[test_df['label'] == ci].iloc[0] for ci in range(len(CLASSES))]
    cam = GradCAM(model=model, target_layers=target_layers_for(model, backbone))
    fig, axes = plt.subplots(2, len(CLASSES), figsize=(len(CLASSES) * 2.1, 4.4))
    for j, row in enumerate(rows):
        rgb, tens = load_for_cam(row['path'])
        pred = int(model(tens).softmax(1).argmax(1).item())
        gcam = cam(input_tensor=tens, targets=[ClassifierOutputTarget(pred)])[0]
        overlay = show_cam_on_image(rgb, gcam, use_rgb=True)
        axes[0, j].imshow(rgb);     axes[0, j].axis('off')
        axes[0, j].set_title(f"real: {CLASSES[row['label']]}", fontsize=8)
        axes[1, j].imshow(overlay); axes[1, j].axis('off')
        axes[1, j].set_title(f"pred: {CLASSES[pred]}", fontsize=8)
    axes[0, 0].set_ylabel('original'); axes[1, 0].set_ylabel('Grad-CAM')
    plt.suptitle(f'Grad-CAM — {run_name}'); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'gradcam_{run_name}.png'), dpi=120); plt.show()

for name, (m, b) in models.items():
    gradcam_grid(m, b, name)

## 5. Galeria de erros

Exemplos classificados errado (real × predito) — útil para ilustrar onde os modelos tropeçam (tipicamente `mel` ↔ `nv`).

In [ ]:
def error_gallery(model, run_name, n=8):
    yt, yp = predict_loader(model, test_loader, tta=True)
    wrong = np.where(yt != yp)[0]
    if len(wrong) == 0:
        print('Sem erros (?)'); return
    rng = np.random.default_rng(SEED)
    sel = rng.choice(wrong, size=min(n, len(wrong)), replace=False)
    cols = 4; rows = int(np.ceil(len(sel) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.9))
    axes = np.atleast_1d(axes).ravel()
    for ax, idx in zip(axes, sel):
        row = test_df.iloc[idx]
        ax.imshow(Image.open(row['path']).convert('RGB')); ax.axis('off')
        ax.set_title(f"real: {CLASSES[yt[idx]]}\npred: {CLASSES[yp[idx]]}", fontsize=8)
    for ax in axes[len(sel):]: ax.axis('off')
    plt.suptitle(f'Erros — {run_name}'); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'erros_{run_name}.png'), dpi=120); plt.show()

# escolha o modelo para a galeria (troque a chave se quiser o outro)
chosen = list(models.keys())[0]
error_gallery(models[chosen][0], chosen, n=8)

## 6. Notas para o relatório

- **t-SNE:** comente a sobreposição `mel`↔`nv` — ela explica visualmente a principal confusão das matrizes.
- **Grad-CAM:** compare os dois modelos; verifique se o foco cai na lesão. Se cair em pelos/régua/tinta, isso é um achado importante (viés de artefato) e mostra maturidade na análise.
- **Galeria de erros:** escolha 1–2 casos emblemáticos para o texto (ex.: melanoma confundido com nevo).
- **Reprodutibilidade:** os checkpoints em `CKPT_DIR` permitem regenerar qualquer figura sem retreinar. Aponte `CKPT_DIR` para o Drive se quiser guardá-los entre sessões.